# 01 — Stage 7 completed-data selected-model replay

Build strictly causal features from the completed producer exports and preserved historical games, then replay the exact installed winners. No model search or fitting. Whole-game cohorts are frozen before scoring; previously reviewed games remain development. The locked retrospective unseen cohort is distinct from future prospective confirmation. Prediction results alone do not establish Hero strategy EV.

Run all cells in order. Use CPU High-RAM, Python 3.12. Preserve failed/checkpoint outputs; resume this exact package. Returned ZIPs require external byte readback and strict import.

Revision r2 repairs one exported timestamp tie. The accepted raw-source ZIP and evaluation protocol remain unchanged; only preparation code and its plan identity change.

In [ ]:
import os,sys,json,hashlib,pathlib,subprocess,zipfile,time,math,shutil,uuid
assert sys.version_info[:2]==(3,12),sys.version
for key in ('OMP_NUM_THREADS','OPENBLAS_NUM_THREADS','MKL_NUM_THREADS','NUMEXPR_NUM_THREADS'):os.environ[key]='1'
mem=dict(line.split(':',1) for line in pathlib.Path('/proc/meminfo').read_text().splitlines())
host_gb=int(mem['MemTotal'].split()[0])*1024/1e9
assert host_gb>=48,f'CPU High-RAM required; observed {host_gb:.2f} GB'
capacity=len(os.sched_getaffinity(0)) if hasattr(os,'sched_getaffinity') else os.cpu_count()
quota=pathlib.Path('/sys/fs/cgroup/cpu.max')
if quota.exists():
    q,period=quota.read_text().split()
    if q!='max':capacity=min(capacity,max(1,math.floor(int(q)/int(period))))
REQUESTED_WORKERS='Auto' # Positive custom count 1 or 2 is supported.
workers=min(2,capacity) if REQUESTED_WORKERS=='Auto' else min(int(REQUESTED_WORKERS),2,capacity)
assert workers>=1
from google.colab import drive
if not pathlib.Path('/content/drive/MyDrive').is_dir():drive.mount('/content/drive')
DRIVE_ROOT=pathlib.Path('/content/drive/MyDrive/Poker Assistant Model/Stage 7 - Empirical Assistant Layers')
print({'python':sys.version,'host_ram_gb':host_gb,'effective_cpus':capacity,'workers':workers,'inner_threads':1})


## Frozen dependencies
Inference and feature work run in fresh subprocesses after installation.

In [ ]:
requirements=['numpy==2.2.6','pandas==2.3.3','pyarrow==21.0.0','joblib==1.5.2','scikit-learn==1.7.2','lightgbm==4.6.0','catboost==1.2.10','xgboost==3.3.0','threadpoolctl==3.6.0']
subprocess.run([sys.executable,'-m','pip','install','--quiet',*requirements],check=True)


## Authenticate direct inputs
ZIP membership, safe paths, byte counts and SHA-256 are checked before importing packaged code or deserializing models. Copy once to local scratch; durable checkpoints stay in this notebook’s own directory.

In [ ]:
EXPECTED_PACKAGE_SHA256='b76b9520e48c42dabb842805904a44b3f4c2473aa6d9e70f23983f99c071e212'
SOURCE=DRIVE_ROOT/'inputs/stage6_empirical_v2_completed_r3b.zip'
def file_sha(path):
    with path.open('rb') as handle:return hashlib.file_digest(handle,'sha256').hexdigest()
assert file_sha(SOURCE)==EXPECTED_PACKAGE_SHA256,'Package mismatch'
WORKSPACE=pathlib.Path('/content/stage6_empirical_v2_'+EXPECTED_PACKAGE_SHA256[:12]);WORKSPACE.mkdir(exist_ok=True)
LOCAL=WORKSPACE/'input.zip'
if not LOCAL.exists():
    partial=LOCAL.with_name(LOCAL.name+'.partial-'+uuid.uuid4().hex)
    shutil.copyfile(SOURCE,partial)
    assert partial.stat().st_size==SOURCE.stat().st_size and file_sha(partial)==EXPECTED_PACKAGE_SHA256
    partial.replace(LOCAL)
assert file_sha(LOCAL)==EXPECTED_PACKAGE_SHA256
with zipfile.ZipFile(LOCAL) as bundle:
    names=bundle.namelist();assert len(names)==len(set(names))
    manifest=json.loads(bundle.read('package_manifest.json'))
    assert set(names)==set(manifest['members'])|{'package_manifest.json'}
    for name,identity in manifest['members'].items():
        relative=pathlib.PurePosixPath(name)
        assert not relative.is_absolute() and '..' not in relative.parts and '\\' not in name and ':' not in name
        target=WORKSPACE/name;target.parent.mkdir(parents=True,exist_ok=True)
        if not target.exists():
            partial=target.with_name(target.name+'.partial-'+uuid.uuid4().hex)
            with bundle.open(name) as source,partial.open('xb') as output:shutil.copyfileobj(source,output,8*1024*1024)
            assert partial.stat().st_size==identity['bytes'] and file_sha(partial)==identity['sha256'],name
            partial.replace(target)
        assert target.stat().st_size==identity['bytes'] and file_sha(target)==identity['sha256'],name
PLAN=WORKSPACE/manifest['plan_relative_path'];assert file_sha(PLAN)==manifest['plan_sha256']
DURABLE=DRIVE_ROOT/'runtime_state'/('88_empirical_v2_'+manifest['plan_sha256'][:16])
PREPARED=DURABLE/'prepared';SCORES=DURABLE/'scores'
ENTRY=WORKSPACE/'Poker Assistant Model/Stage 6 - Qualified Layers Improvement/src'
def execute(command):
    process=subprocess.Popen(command,cwd=WORKSPACE,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    for line in process.stdout:print(line.rstrip(),flush=True)
    assert process.wait()==0,'Execution failed; preserve exact evidence and checkpoints'
print({'authenticated_members':len(manifest['members']),'plan_sha256':manifest['plan_sha256'],'durable':str(DURABLE)})


## Authenticate timestamp repair
Accept exported decision/completion ties while keeping completed history strictly earlier than each query. Reuse unchanged cached source and model bytes in a separate successor workspace.

In [ ]:
# Authenticated two-member repair; retain the original input cache and timestamps.
OVERLAY=DRIVE_ROOT/'inputs/stage7_01_timestamp_code_plan_r2.zip'
EXPECTED_OVERLAY={'bytes': 24959, 'sha256': 'af93af7fd78fc171c9d7eb57f97cab9698ee9af14f0036cc58ba2a8ae12d94d6'}
assert OVERLAY.stat().st_size==EXPECTED_OVERLAY['bytes'] and file_sha(OVERLAY)==EXPECTED_OVERLAY['sha256']
with zipfile.ZipFile(OVERLAY) as overlay_bundle:
    helper_source=overlay_bundle.read('stage7_completed_overlay.py')
    assert hashlib.sha256(helper_source).hexdigest()=='6d440f39aff71adade9b00f0f84b40a136939dd594a379917e66fb3efde04f5b'
overlay_module={'__name__':'stage7_completed_overlay'}
exec(compile(helper_source,'stage7_completed_overlay.py','exec'),overlay_module)
BASE_WORKSPACE=pathlib.Path('/content/stage6_empirical_v2_'+EXPECTED_PACKAGE_SHA256[:12])
with zipfile.ZipFile(BASE_WORKSPACE/'input.zip') as base_bundle:
    base_manifest=json.loads(base_bundle.read('package_manifest.json'))
WORKSPACE,manifest=overlay_module['activate'](BASE_WORKSPACE,pathlib.Path('/content/stage7_01_r2_'+'af93af7fd78f'),base_manifest,OVERLAY,EXPECTED_OVERLAY)
PLAN=WORKSPACE/manifest['plan_relative_path']
DURABLE=DRIVE_ROOT/'runtime_state'/('88_empirical_v2_'+manifest['plan_sha256'][:16])
PREPARED=DURABLE/'prepared';SCORES=DURABLE/'scores'
ENTRY=WORKSPACE/'Poker Assistant Model/Stage 6 - Qualified Layers Improvement/src'
print({'status':'AUTHENTICATED_TIMESTAMP_REPAIR','plan_sha256':manifest['plan_sha256'],'workspace':str(WORKSPACE),'durable':str(DURABLE),'unchanged_source_members':len(manifest['members'])-2})


## Causal source preparation
All older games remain eligible history. Public observations enter features only after their hand completes and before the decision. Monetary all-in calls retain their true aggressor; exact identities are remapped without using names. Missing newer reveal labels remain unobserved. A completed preparation resumes by its receipt.

In [ ]:
execute([sys.executable,str(ENTRY/'stage6_empirical_preparation_v2.py'),'--plan',str(PLAN),'--output',str(PREPARED)])
preparation=json.loads((PREPARED/'preparation_receipt.json').read_text())
print({key:preparation[key] for key in ('decisions','history_rows','history_games','decision_games','quarantined_decisions')})


## Frozen model replay
Independent layer jobs have disjoint checkpoints, at most two processes and one inner thread. Completed layers resume without refitting or rescoring. The same causal rows feed each paired comparison.

In [ ]:
execute([sys.executable,str(ENTRY/'stage6_empirical_evaluation_v2.py'),'--snapshot-root',str(PREPARED),'--output',str(SCORES),'--workers',str(workers)])
report=json.loads((SCORES/'report.json').read_text())
print({'rows':report['rows'],'games':report['game_clusters'],'cohorts':report['cohort_rows'],'design':report['confirmation_design']})
for name,layer in report['layers'].items():print(name,layer['qualification'])


## Publish direct results
Publish the complete prepared source, protocol, per-layer predictions and receipts, then the embedded manifest last. No automatic promotion. This exact result is a prerequisite for downstream empirical comparisons.

In [ ]:
RETURN=DRIVE_ROOT/'returned_results'/('88_Stage6_Completed_Replay_'+report['run_id'][:16]+'.zip')
members={}
for path in PREPARED.iterdir():
    if path.is_file() and path.suffix!='.partial':members['prepared/'+path.name]=path
for path in SCORES.iterdir():
    if path.is_file() and (path.name.endswith('.predictions.npz') or path.name.endswith('.receipt.json') or path.name in ('report.json','report_receipt.json','run_identity.json')):members['scores/'+path.name]=path
result_manifest={'schema_version':'stage6_empirical_completed_result_v2','run_id':report['run_id'],'package_sha256':EXPECTED_PACKAGE_SHA256,'plan_sha256':manifest['plan_sha256'],'members':{name:{'bytes':path.stat().st_size,'sha256':file_sha(path)} for name,path in sorted(members.items())}}
if not RETURN.exists():
    partial=RETURN.with_name(RETURN.name+'.partial-'+uuid.uuid4().hex)
    with zipfile.ZipFile(partial,'x',compression=zipfile.ZIP_DEFLATED) as bundle:
        for name,path in sorted(members.items()):bundle.write(path,name)
        bundle.writestr('result_manifest.json',json.dumps(result_manifest,sort_keys=True,separators=(',',':')))
    partial.replace(RETURN)
with zipfile.ZipFile(RETURN) as bundle:
    assert len(bundle.namelist())==len(set(bundle.namelist()))
    assert json.loads(bundle.read('result_manifest.json'))==result_manifest
    for name,identity in result_manifest['members'].items():
        with bundle.open(name) as handle:assert hashlib.file_digest(handle,'sha256').hexdigest()==identity['sha256']
print({'result':str(RETURN),'bytes':RETURN.stat().st_size,'sha256':file_sha(RETURN),'status':'PUBLISHED_AWAITING_EXTERNAL_READBACK_AND_STRICT_IMPORT'})
